# 3.3 Anwendung: Wärmeübertragung in einer Mehrschichtwand

Bisher haben wir LGS an einem Obstmarkt-Beispiel geübt. Jetzt wenden wir
dasselbe Werkzeug auf ein Problem aus dem Maschinenbau an: eine Außenwand
aus drei Schichten mit unterschiedlichen Wärmedurchgangswiderständen. *Wie
hoch ist die Temperatur an den Grenzflächen, und wie groß ist der
Wärmestrom?*

Wir werden sehen, dass der Weg von den physikalischen Gleichungen zur
Matrix immer gleich ist: Unbekannte auf die linke Seite, bekannte Größen
auf die rechte.

## Lernziele

* [ ] Sie können physikalische Gleichgewichtsbedingungen in ein LGS
  $\mathbf{A} \cdot \vec{x} = \vec{b}$ überführen.
* [ ] Sie können die Matrix $\mathbf{A}$ und den Vektor $\vec{b}$ aus
  umgeformten Gleichungen ablesen.
* [ ] Sie können das LGS lösen und das Ergebnis physikalisch interpretieren.

## Das physikalische Modell

Wir betrachten eine Wand aus drei Schichten A, B, C mit den thermischen
Widerständen $R_A$, $R_B$, $R_C$ in K/W. Links herrscht die Temperatur
$T_{LA}$, rechts $T_{CR}$.

![Querschnitt einer dreischichtigen Wand mit Temperaturprofil und Wärmestrom](pics/waermeuebertragung_mehrschichtwand.svg)

Temperaturprofil einer Mehrschichtwand im stationären Zustand (schematische
Darstellung bei gleicher geometrischer Schichtdicke). Da der Wärmestrom durch
alle Schichten gleich ist, ist die Steigung des Temperaturprofils proportional
zum thermischen Widerstand der jeweiligen Schicht, in Schicht C am steilsten, in
Schicht B am flachsten. (Quelle: eigene Abbildung; Lizenz [CC BY-SA
4.0](https://creativecommons.org/licenses/by-sa/4.0))

Im **stationären Zustand** ist der Wärmestrom $Q$ durch alle Schichten
gleich. Das **Wärmeübertragungsgesetz** (analog zum Ohmschen Gesetz) lautet
für jede Schicht:

$$Q = \frac{\Delta T_i}{R_i}$$

Das liefert drei Gleichungen mit drei Unbekannten ($T_{AB}$, $T_{BC}$, $Q$):

$$\frac{T_{LA} - T_{AB}}{R_A} = Q \qquad (1)$$

$$\frac{T_{AB} - T_{BC}}{R_B} = Q \qquad (2)$$

$$\frac{T_{BC} - T_{CR}}{R_C} = Q \qquad (3)$$

## Von den Gleichungen zur Matrixform

Wir bringen alle Unbekannten auf die linke Seite. Dafür multiplizieren wir
jede Gleichung mit $R_i$ und ordnen um:

$$T_{AB} + R_A \cdot Q = T_{LA} \qquad (1')$$

$$-T_{AB} + T_{BC} + R_B \cdot Q = 0 \qquad (2')$$

$$-T_{BC} + R_C \cdot Q = -T_{CR} \qquad (3')$$

Jetzt lesen wir die Koeffizientenmatrix zeilenweise ab.
Der Lösungsvektor ist $\vec{x} = (T_{AB},\, T_{BC},\, Q)^T$:

$$\underbrace{\begin{pmatrix}
+1 &  0 & R_A \\
-1 & +1 & R_B \\
 0 & -1 & R_C
\end{pmatrix}}_{\mathbf{A}}
\cdot
\underbrace{\begin{pmatrix} T_{AB} \\ T_{BC} \\ Q \end{pmatrix}}_{\vec{x}}
=
\underbrace{\begin{pmatrix} T_{LA} \\ 0 \\ -T_{CR} \end{pmatrix}}_{\vec{b}}$$

Jede umgeformte Gleichung liefert eine Zeile von $\mathbf{A}$ und einen
Eintrag in $\vec{b}$. Der Koeffizient der $j$-ten Unbekannten in der
$i$-ten Gleichung steht in $A_{ij}$. Unbekannte, die nicht vorkommen,
erhalten den Koeffizienten 0.

## Implementierung und Lösung

In [ ]:
import numpy as np

# ---------- gegebene Größen ----------
R_A  = 0.5   # thermischer Widerstand Schicht A in K/W
R_B  = 0.3   # thermischer Widerstand Schicht B in K/W
R_C  = 0.7   # thermischer Widerstand Schicht C in K/W
T_LA = 300.  # Temperatur linke Seite in K
T_CR = 310.  # Temperatur rechte Seite in K

# ---------- Koeffizientenmatrix aufstellen ----------
# Unbekannte: x = [T_AB, T_BC, Q]
#
# Gleichung (1') aus  (T_LA - T_AB)/R_A = Q
#   => T_AB + 0*T_BC + R_A*Q = T_LA
#   => Zeile 1: [+1,  0, R_A]  |  b[0] = T_LA
#
# Gleichung (2') aus  (T_AB - T_BC)/R_B = Q
#   => -T_AB + T_BC + R_B*Q = 0
#   => Zeile 2: [-1, +1, R_B]  |  b[1] = 0
#
# Gleichung (3') aus  (T_BC - T_CR)/R_C = Q
#   => 0*T_AB - T_BC + R_C*Q = -T_CR
#   => Zeile 3: [ 0, -1, R_C]  |  b[2] = -T_CR
A = np.array([
    [+1.,  0., R_A],
    [-1., +1., R_B],
    [ 0., -1., R_C],
])
b = np.array([T_LA, 0., -T_CR])

# ---------- Lösbarkeit prüfen (wie in Kapitel 3.1) ----------
det_A = np.linalg.det(A)
print(f'Determinante: {det_A:.4f}')
if np.isclose(det_A, 0.):
    print('Keine eindeutige Lösung.')
else:
    print('Eindeutige Lösung vorhanden.')

# ---------- Lösen und Ergebnis ausgeben (wie in Kapitel 3.2) ----------
x = np.linalg.solve(A, b)
T_AB, T_BC, Q = x   # Ergebnis in drei Variablen entpacken

print(f'\nT_AB = {T_AB:.2f} K   (Grenzfläche A-B)')
print(f'T_BC = {T_BC:.2f} K   (Grenzfläche B-C)')
print(f'Q    = {Q:.4f} W   (Wärmestrom)')

# ---------- Probe ----------
print('\nProbe bestanden:', np.allclose(A @ x, b))

Das negative Vorzeichen von $Q$ ist kein Fehler: in unserem Ansatz zeigt
die positive Richtung von links nach rechts, aber die Wärme fließt von der
wärmeren rechten Seite ($T_{CR} = 310$ K) zur kälteren linken Seite
($T_{LA} = 300$ K). Die Abbildung zeigt diesen physikalischen Fluss von
rechts nach links. Beides ist konsistent.

Zur Kontrolle berechnen wir noch die Temperaturdifferenz über jede Schicht.
Schicht C hat den größten Widerstand und sollte daher den größten
Temperatursprung liefern, genauso wie der größte Widerstand in einem
elektrischen Schaltkreis den größten Spannungsabfall erzeugt.

In [ ]:
# Temperaturdifferenzen über jede Schicht
# Delta_i = R_i * |Q|: je größer der Widerstand, desto größer der Sprung
delta_A = T_AB - T_LA
delta_B = T_BC - T_AB
delta_C = T_CR - T_BC

print(f'Temperaturdiff. Schicht A (R={R_A} K/W): {delta_A:.2f} K')
print(f'Temperaturdiff. Schicht B (R={R_B} K/W): {delta_B:.2f} K')
print(f'Temperaturdiff. Schicht C (R={R_C} K/W): {delta_C:.2f} K')
print(f'Summe (muss T_CR - T_LA = {T_CR - T_LA:.1f} K ergeben): '
      f'{delta_A + delta_B + delta_C:.2f} K')

### Mini-Übung 1

Verständnisfragen (ohne Code):

1. In der Koeffizientenmatrix steht in Zeile 2, Spalte 1 der Wert $-1$.
   Aus welcher physikalischen Gleichung stammt dieser Eintrag, und warum
   ist er negativ? Begründen Sie in eigenen Worten, ohne Code auszuführen.
2. Warum ist $b[1] = 0$? Was bedeutet das physikalisch?

Rechenaufgabe:

Eine Kühlhauswand hält die Innentemperatur bei $T_{\text{innen}} = 268$ K
(−5 °C), während außen $T_{\text{außen}} = 293$ K (20 °C) herrschen.

| Schicht | Material           | R (K/W) |
|---------|--------------------|---------|
| A       | Beton              | 0.2     |
| B       | Polyurethan-Schaum | 1.8     |
| C       | Stahlblech         | 0.05    |

Die linke Seite ist innen ($T_{LA} = T_{\text{innen}}$), die rechte außen
($T_{CR} = T_{\text{außen}}$). Das LGS hat dieselbe Struktur wie oben, nur
mit anderen Zahlenwerten.

3. Legen Sie `A` und `b` mit den neuen Werten an, lösen Sie das System und
   geben Sie $T_{AB}$, $T_{BC}$ und $Q$ aus.
4. Welche Schicht hat den größten Temperaturabfall? Ist das plausibel?

In [ ]:
# Hier Ihren Code eingeben

## Zusammenfassung und Ausblick

Das Vorgehen ist immer dasselbe: physikalische Gleichungen umformen, alle
Unbekannten nach links, alle bekannten Größen nach rechts, dann
$\mathbf{A}$ und $\vec{b}$ ablesen. `np.linalg.solve` liefert die Lösung,
`np.allclose` sichert sie ab.

Dieses Muster lässt sich auf beliebig komplexe Systeme übertragen. Im
nächsten Kapitel sammeln wir Übungsaufgaben zu LGS, bevor wir es in
Kapitel 3.5 auf elektrische Netzwerke anwenden.